In [2]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk
import torch, gc

In [3]:
MODEL_PATH   = r"C:\datasets\llm-data\models\phi3-base"
DATASET_PATH = r"C:\datasets\llm-data\pretrain\tokenized_dataset"
OUTPUT_PATH  = r"C:\datasets\llm-data\models\phi3-pretrained"

gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")


Free VRAM: 4.96 GB


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0},
    local_files_only=True
)
model = prepare_model_for_kbit_training(model)

c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\utils\modeling.py:416: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  new_value = value.to(device)


In [8]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False)

In [9]:
train_dataset = load_from_disk(f"{DATASET_PATH}/train")
val_dataset   = load_from_disk(f"{DATASET_PATH}/val")

In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=False,    # ← changed to False
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
)

In [17]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    max_seq_length=256,
    peft_config=lora_config,)

In [18]:
trainer.train()


  0%|          | 0/295 [00:00<?, ?it/s]

{'loss': 1.127, 'grad_norm': 0.06297249346971512, 'learning_rate': 0.0002, 'epoch': 0.17}
{'loss': 1.137, 'grad_norm': 0.09488288313150406, 'learning_rate': 0.00015918367346938776, 'epoch': 0.34}
{'loss': 1.128, 'grad_norm': 0.08122919499874115, 'learning_rate': 0.00011836734693877552, 'epoch': 0.51}
{'loss': 1.0848, 'grad_norm': 0.08073864877223969, 'learning_rate': 7.755102040816327e-05, 'epoch': 0.68}


  0%|          | 0/250 [00:00<?, ?it/s]

{'eval_runtime': 246.9541, 'eval_samples_per_second': 1.012, 'eval_steps_per_second': 1.012, 'epoch': 0.68}


c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\Ayush\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


{'loss': 1.1158, 'grad_norm': 0.08776272088289261, 'learning_rate': 3.673469387755102e-05, 'epoch': 0.84}
{'train_runtime': 13851.0363, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.021, 'train_loss': 1.1149397251969677, 'epoch': 1.0}


TrainOutput(global_step=295, training_loss=1.1149397251969677, metrics={'train_runtime': 13851.0363, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.021, 'total_flos': 5.399960535171072e+16, 'train_loss': 1.1149397251969677, 'epoch': 0.996832101372756})

In [19]:
model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

('C:\\datasets\\llm-data\\models\\phi3-pretrained\\tokenizer_config.json',
 'C:\\datasets\\llm-data\\models\\phi3-pretrained\\special_tokens_map.json',
 'C:\\datasets\\llm-data\\models\\phi3-pretrained\\tokenizer.model',
 'C:\\datasets\\llm-data\\models\\phi3-pretrained\\added_tokens.json',
 'C:\\datasets\\llm-data\\models\\phi3-pretrained\\tokenizer.json')